### update the definition from the new and old notes with the two_words_in_furigana_field tag using AI (chatGPT API for example. Or free Ai (even better))

In [ ]:
from cloze import invoke


deck = "Japanese Media::Youtube::Konnichiwa My Dude Japanese Podcast::ジブリ愛が止まらない | 日本語ポッドキャスト EP299"
tags = ["two_words_in_furigana_field"]


note_ids = invoke(
    "findNotes",
    query='deck:"Japanese Media::Youtube::Konnichiwa My Dude Japanese Podcast::ジブリ愛が止まらない | 日本語ポッドキャスト EP299"'
)

#get information about those notes, to find the RTK card ids
notes_info = invoke(
    "notesInfo",
    notes=note_ids['result']
)



In [5]:
# AI DEFINITION UPDATE FOR TAGGED NOTES
# Updates definition text for notes tagged: two_words_in_furigana_field
# Provider options:
# - provider = "openai"  (requires OPENAI_API_KEY)
# - provider = "ollama"  (free/local, requires Ollama running)

import os
import json
import requests

provider = "ollama"            # "openai" or "ollama"
openai_model = "gpt-4o-mini"
ollama_model = "qwen2.5:7b"    # change if needed, e.g. "llama3.1:8b"
dry_run = True                  # set False to write updates to Anki
limit = 50                      # max notes to process this run

TARGET_TAG = "two_words_in_furigana_field"


def ai_rewrite_definition(old_definition, lemma, furigana, cloze, notes_field, word_definition):
    prompt = f"""You are improving Japanese vocabulary flashcard definitions.

Return only one concise definition sentence/line.
Keep the meaning accurate and learner-friendly.
If old definition is already good, lightly clean it.
Do not include markdown, bullets, or explanations.

Lemma: {lemma}
Furigana: {furigana}
Cloze sentence: {cloze}
Old definition: {old_definition}
Notes field: {notes_field}
Word Definition field: {word_definition}
"""

    if provider == "openai":
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            raise RuntimeError("OPENAI_API_KEY is not set")

        r = requests.post(
            "https://api.openai.com/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
            },
            json={
                "model": openai_model,
                "messages": [
                    {"role": "system", "content": "Rewrite definitions clearly and briefly."},
                    {"role": "user", "content": prompt},
                ],
                "temperature": 0.2,
            },
            timeout=60,
        )
        r.raise_for_status()
        data = r.json()
        return data["choices"][0]["message"]["content"].strip()

    if provider == "ollama":
        r = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": ollama_model,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0.2},
            },
            timeout=120,
        )
        r.raise_for_status()
        data = r.json()
        return data.get("response", "").strip()

    raise ValueError(f"Unknown provider: {provider}")


note_ids = invoke("findNotes", query=f'tag:{TARGET_TAG}')
if note_ids.get("error"):
    raise RuntimeError(note_ids["error"])

ids = note_ids.get("result", [])[:limit]
print(f"Found {len(note_ids.get('result', []))} notes with tag '{TARGET_TAG}'. Processing {len(ids)}.")

if not ids:
    print("No notes to process.")
else:
    notes_info = invoke("notesInfo", notes=ids)
    if notes_info.get("error"):
        raise RuntimeError(notes_info["error"])

    updated = 0
    skipped = 0

    for note in notes_info.get("result", []):
        note_id = note["noteId"]
        fields = note.get("fields", {})

        lemma = fields.get("Lemma", {}).get("value", "").strip()
        furigana = fields.get("Furigana", {}).get("value", "").strip()
        cloze = fields.get("Cloze", {}).get("value", "").strip()
        notes_field = fields.get("Notes", {}).get("value", "").strip()
        word_definition = fields.get("Word Definition", {}).get("value", "").strip()

        # Prefer Notes as target; fallback to Word Definition if Notes missing.
        if "Notes" in fields:
            target_field = "Notes"
            old_definition = notes_field
        elif "Word Definition" in fields:
            target_field = "Word Definition"
            old_definition = word_definition
        else:
            print(f"Skip {note_id}: no Notes/Word Definition field")
            skipped += 1
            continue

        try:
            new_definition = ai_rewrite_definition(
                old_definition=old_definition,
                lemma=lemma,
                furigana=furigana,
                cloze=cloze,
                notes_field=notes_field,
                word_definition=word_definition,
            )
        except Exception as e:
            print(f"Skip {note_id}: AI call failed -> {e}")
            skipped += 1
            continue

        if not new_definition or new_definition == old_definition:
            print(f"No change {note_id}")
            skipped += 1
            continue

        print(f"Update {note_id} ({target_field})")
        print(f"  OLD: {old_definition[:120]}")
        print(f"  NEW: {new_definition[:120]}")

        if not dry_run:
            res = invoke(
                "updateNoteFields",
                note={
                    "id": note_id,
                    "fields": {target_field: new_definition},
                },
            )
            if res.get("error"):
                print(f"  Failed update {note_id}: {res['error']}")
                skipped += 1
                continue

        updated += 1

    print(json.dumps({"updated": updated, "skipped": skipped, "dry_run": dry_run}, ensure_ascii=False))



Found 290 notes with tag 'two_words_in_furigana_field'. Processing 50.
Skip 1772362079082: AI call failed -> HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 61] Connection refused"))
Skip 1772362100369: AI call failed -> HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 61] Connection refused"))
Skip 1772362116815: AI call failed -> HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 61] Connection refused"))
Skip 1772362127337: AI call failed -> HTTPConnectionPool(host='localhost', port=11434)